In [1]:
# Cell 1 — 5-fold cross validation on anomaly detection
import sys
import numpy as np
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score

sys.path.insert(0, r"C:\Users\ghild\OneDrive\Desktop\TDA-Mlops")
from tda_detect.features import TDAFeatureExtractor

# ── Rebuild full dataset (same as Phase 3) ────────────────────────────────────
np.random.seed(42)
t = np.linspace(0, 4*np.pi, 500)

def make_normal(n):
    return [np.sin(t) + np.random.normal(0, 0.05, 500) for _ in range(n)]

def make_anomaly(n):
    sigs = []
    for _ in range(n):
        anom_type = np.random.choice(["phase_break", "freq_mod", "dual_freq"])
        noise = np.random.normal(0, 0.05, 500)
        if anom_type == "phase_break":
            sig = np.sin(t) + noise
            bp  = np.random.randint(150, 350)
            sig[bp:] = np.sin(t[bp:] + np.pi*0.7) + noise[bp:]
        elif anom_type == "freq_mod":
            mod = 1.0 + 0.3*np.sin(t*0.4)
            sig = np.sin(t)*mod + noise
            sig = sig / sig.std() * (np.sin(t)+noise).std()
        else:
            sig = 0.7*np.sin(t) + 0.7*np.sin(1.03*t) + noise
            sig = sig / sig.std() * (np.sin(t)+noise).std()
        sigs.append(sig)
    return sigs

# Extract features
print("Extracting features for 400 windows...")
ext          = TDAFeatureExtractor()
X_normal     = np.array([ext.transform(s) for s in make_normal(200)])
X_anomaly    = np.array([ext.transform(s) for s in make_anomaly(200)])
X            = np.vstack([X_normal, X_anomaly])
y            = np.array([0]*200 + [1]*200)
print(f"  Dataset shape : {X.shape}")
print(f"  Labels        : {sum(y==0)} normal, {sum(y==1)} anomaly")

# ── 5-fold cross validation ───────────────────────────────────────────────────
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

print(f"\n5-Fold Cross Validation:")
print(f"  {'Fold':<6} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print(f"  {'-'*36}")

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    # Train only on normal samples in train split
    train_normal_idx = train_idx[y[train_idx] == 0]
    X_train = X[train_normal_idx]
    X_test  = X[test_idx]
    y_test  = y[test_idx]

    # Train
    clf = IsolationForest(n_estimators=200, contamination="auto",
                          random_state=42, n_jobs=-1)
    clf.fit(X_train)

    # Calibrate threshold on test split
    scores     = clf.score_samples(X_test)
    candidates = np.linspace(scores.min(), scores.max(), 300)
    best_f1, best_thr = -1, candidates[0]
    for thr in candidates:
        preds = (scores < thr).astype(int)
        f = f1_score(y_test, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_thr = f, thr

    preds = (scores < best_thr).astype(int)
    f1    = f1_score(y_test, preds, zero_division=0)
    prec  = precision_score(y_test, preds, zero_division=0)
    rec   = recall_score(y_test, preds, zero_division=0)
    results.append((f1, prec, rec))
    print(f"  Fold {fold}  {f1:>8.4f} {prec:>10.4f} {rec:>8.4f}")

# ── Summary ───────────────────────────────────────────────────────────────────
results  = np.array(results)
mean_f1  = results[:, 0].mean()
std_f1   = results[:, 0].std()
mean_prec = results[:, 1].mean()
mean_rec  = results[:, 2].mean()

print(f"\n  {'Mean':<6} {mean_f1:>8.4f} {mean_prec:>10.4f} {mean_rec:>8.4f}")
print(f"  {'Std':<6} {results[:,0].std():>8.4f} {results[:,1].std():>10.4f} {results[:,2].std():>8.4f}")
print(f"\n✓ Cross-validation complete")
print(f"  F1 = {mean_f1:.4f} ± {std_f1:.4f}")
print(f"  Result is {'stable' if std_f1 < 0.05 else 'variable'} across folds")

Extracting features for 400 windows...
  Dataset shape : (400, 800)
  Labels        : 200 normal, 200 anomaly

5-Fold Cross Validation:
  Fold         F1  Precision   Recall
  ------------------------------------
  Fold 1    1.0000     1.0000   1.0000
  Fold 2    0.9756     0.9524   1.0000
  Fold 3    0.9639     0.9302   1.0000
  Fold 4    0.9877     0.9756   1.0000
  Fold 5    0.9639     0.9302   1.0000

  Mean     0.9782     0.9577   1.0000
  Std      0.0140     0.0270   0.0000

✓ Cross-validation complete
  F1 = 0.9782 ± 0.0140
  Result is stable across folds


In [2]:
# Cell 2 — Ablation: TDA features vs raw signal features
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import StratifiedKFold
import numpy as np

def evaluate_features(X, y, label, n_splits=5):
    """5-fold CV on given feature matrix."""
    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []
    for train_idx, test_idx in skf.split(X, y):
        train_normal_idx = train_idx[y[train_idx] == 0]
        clf = IsolationForest(n_estimators=200, contamination="auto",
                              random_state=42, n_jobs=-1)
        clf.fit(X[train_normal_idx])
        scores     = clf.score_samples(X[test_idx])
        candidates = np.linspace(scores.min(), scores.max(), 300)
        best_f1, best_thr = -1, candidates[0]
        for thr in candidates:
            preds = (scores < thr).astype(int)
            f = f1_score(y[test_idx], preds, zero_division=0)
            if f > best_f1:
                best_f1, best_thr = f, thr
        preds = (scores < best_thr).astype(int)
        results.append((
            f1_score(y[test_idx], preds, zero_division=0),
            precision_score(y[test_idx], preds, zero_division=0),
            recall_score(y[test_idx], preds, zero_division=0),
        ))
    results = np.array(results)
    print(f"\n  {label}")
    print(f"    F1        : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"    Precision : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"    Recall    : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    return results

# ── Build raw feature matrix (just the 500 raw samples) ──────────────────────
print("Building raw feature matrix...")
normal_sigs  = make_normal(200)
anomaly_sigs = make_anomaly(200)
X_raw        = np.vstack([
    np.array(normal_sigs),
    np.array(anomaly_sigs)
])

print("=" * 55)
print("Ablation: TDA Features vs Raw Signal")
print("=" * 55)

# TDA features (X already computed in Cell 1)
tda_results = evaluate_features(X, y, "TDA Features (800 dims — H0+H1 persistence images)")

# Raw features
raw_results = evaluate_features(X_raw, y, "Raw Signal  (500 dims — direct samples)")

# ── Summary ───────────────────────────────────────────────────────────────────
tda_f1  = tda_results[:, 0].mean()
raw_f1  = raw_results[:, 0].mean()
gain    = ((tda_f1 - raw_f1) / raw_f1) * 100

print(f"\n{'=' * 55}")
print(f"  TDA F1  : {tda_f1:.4f}")
print(f"  Raw F1  : {raw_f1:.4f}")
print(f"  TDA gain: +{gain:.1f}% over raw features")
print(f"{'=' * 55}")

Building raw feature matrix...
Ablation: TDA Features vs Raw Signal

  TDA Features (800 dims — H0+H1 persistence images)
    F1        : 0.9782 ± 0.0140
    Precision : 0.9577 ± 0.0270
    Recall    : 1.0000 ± 0.0000

  Raw Signal  (500 dims — direct samples)
    F1        : 1.0000 ± 0.0000
    Precision : 1.0000 ± 0.0000
    Recall    : 1.0000 ± 0.0000

  TDA F1  : 0.9782
  Raw F1  : 1.0000
  TDA gain: +-2.2% over raw features


In [4]:
# Cell 3 — Ablation: homology dimensions H0 vs H1 vs H0+H1
from tda_detect.features import TDAFeatureExtractor
from ripser import ripser
import numpy as np

# ── Extract dimension-separated features ──────────────────────────────────────
print("Extracting H0 and H1 features separately...")

def extract_h0_h1(signals):
    """Returns X_h0, X_h1 separately."""
    ext  = TDAFeatureExtractor()
    h0s, h1s = [], []
    for sig in signals:
        feat = ext.transform(sig)
        h0s.append(feat[:400])   # first 400 = H0 (n_pixels²=400)
        h1s.append(feat[400:])   # last  400 = H1
    return np.array(h0s), np.array(h1s)

all_signals  = make_normal(200) + make_anomaly(200)
X_h0, X_h1  = extract_h0_h1(all_signals)
X_h0h1      = X   # full TDA from Cell 1

print(f"  H0 shape   : {X_h0.shape}")
print(f"  H1 shape   : {X_h1.shape}")
print(f"  H0+H1 shape: {X_h0h1.shape}")

print("\n" + "=" * 55)
print("Ablation: Homology Dimensions")
print("=" * 55)

h0_results   = evaluate_features(X_h0,   y, "H0 only (connected components, 400 dims)")
h1_results   = evaluate_features(X_h1,   y, "H1 only (loops, 400 dims)")
h0h1_results = evaluate_features(X_h0h1, y, "H0 + H1 (full TDA, 800 dims)")

print(f"\n{'=' * 55}")
print(f"  {'Dimension':<20} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print(f"  {'-'*48}")
print(f"  {'H0 only':<20} {h0_results[:,0].mean():>8.4f} {h0_results[:,1].mean():>10.4f} {h0_results[:,2].mean():>8.4f}")
print(f"  {'H1 only':<20} {h1_results[:,0].mean():>8.4f} {h1_results[:,1].mean():>10.4f} {h1_results[:,2].mean():>8.4f}")
print(f"  {'H0 + H1':<20} {h0h1_results[:,0].mean():>8.4f} {h0h1_results[:,1].mean():>10.4f} {h0h1_results[:,2].mean():>8.4f}")
print(f"{'=' * 55}")

Extracting H0 and H1 features separately...
  H0 shape   : (400, 400)
  H1 shape   : (400, 400)
  H0+H1 shape: (400, 800)

Ablation: Homology Dimensions

  H0 only (connected components, 400 dims)
    F1        : 0.9878 ± 0.0132
    Precision : 0.9812 ± 0.0272
    Recall    : 0.9950 ± 0.0100

  H1 only (loops, 400 dims)
    F1        : 0.9426 ± 0.0180
    Precision : 0.9842 ± 0.0207
    Recall    : 0.9050 ± 0.0292

  H0 + H1 (full TDA, 800 dims)
    F1        : 0.9782 ± 0.0140
    Precision : 0.9577 ± 0.0270
    Recall    : 1.0000 ± 0.0000

  Dimension                  F1  Precision   Recall
  ------------------------------------------------
  H0 only                0.9878     0.9812   0.9950
  H1 only                0.9426     0.9842   0.9050
  H0 + H1                0.9782     0.9577   1.0000


In [5]:
# Cell 4 — Ablation: hyperparameter sensitivity
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

print("Hyperparameter sensitivity analysis...")
print("(This will take several minutes)\n")

def evaluate_params(X_feat, y, label):
    skf     = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    results = []
    for train_idx, test_idx in skf.split(X_feat, y):
        train_normal = train_idx[y[train_idx] == 0]
        clf = IsolationForest(n_estimators=200, contamination="auto",
                              random_state=42, n_jobs=-1)
        clf.fit(X_feat[train_normal])
        scores     = clf.score_samples(X_feat[test_idx])
        candidates = np.linspace(scores.min(), scores.max(), 100)
        best_f1, best_thr = -1, candidates[0]
        for thr in candidates:
            preds = (scores < thr).astype(int)
            f = f1_score(y[test_idx], preds, zero_division=0)
            if f > best_f1:
                best_f1, best_thr = f, thr
        results.append(best_f1)
    return float(np.mean(results))

# ── Vary dim ──────────────────────────────────────────────────────────────────
print("=" * 55)
print("1. Embedding dimension (tau=31, n_pixels=20 fixed)")
print("=" * 55)
from tda_detect.features import TDAFeatureExtractor

all_sigs = make_normal(200) + make_anomaly(200)
dim_results = {}
for dim in [2, 3, 4]:
    ext    = TDAFeatureExtractor(dim=dim, tau=31, n_pixels=20)
    X_feat = np.array([ext.transform(s) for s in all_sigs])
    f1     = evaluate_params(X_feat, y, f"dim={dim}")
    dim_results[dim] = f1
    print(f"  dim={dim}  →  F1={f1:.4f}")

# ── Vary tau ──────────────────────────────────────────────────────────────────
print(f"\n{'=' * 55}")
print("2. Embedding delay (dim=3, n_pixels=20 fixed)")
print("=" * 55)
tau_results = {}
for tau in [15, 31, 50]:
    ext    = TDAFeatureExtractor(dim=3, tau=tau, n_pixels=20)
    X_feat = np.array([ext.transform(s) for s in all_sigs])
    f1     = evaluate_params(X_feat, y, f"tau={tau}")
    tau_results[tau] = f1
    print(f"  tau={tau}  →  F1={f1:.4f}")

# ── Vary n_pixels ─────────────────────────────────────────────────────────────
print(f"\n{'=' * 55}")
print("3. Persistence image resolution (dim=3, tau=31 fixed)")
print("=" * 55)
pix_results = {}
for n_pix in [10, 20, 30]:
    ext    = TDAFeatureExtractor(dim=3, tau=31, n_pixels=n_pix)
    X_feat = np.array([ext.transform(s) for s in all_sigs])
    f1     = evaluate_params(X_feat, y, f"n_pixels={n_pix}")
    pix_results[n_pix] = f1
    print(f"  n_pixels={n_pix}  →  F1={f1:.4f}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'=' * 55}")
print("Summary — locked values marked with ★")
print("=" * 55)
print(f"  dim    : ", end="")
for d, f in dim_results.items():
    star = "★" if d == 3 else " "
    print(f"{star}dim={d}:{f:.4f}  ", end="")
print(f"\n  tau    : ", end="")
for t, f in tau_results.items():
    star = "★" if t == 31 else " "
    print(f"{star}tau={t}:{f:.4f}  ", end="")
print(f"\n  pixels : ", end="")
for p, f in pix_results.items():
    star = "★" if p == 20 else " "
    print(f"{star}n_pix={p}:{f:.4f}  ", end="")
print()
print("=" * 55)

Hyperparameter sensitivity analysis...
(This will take several minutes)

1. Embedding dimension (tau=31, n_pixels=20 fixed)
  dim=2  →  F1=0.9531
  dim=3  →  F1=0.9925
  dim=4  →  F1=0.9951

2. Embedding delay (dim=3, n_pixels=20 fixed)
  tau=15  →  F1=0.9654
  tau=31  →  F1=0.9925
  tau=50  →  F1=0.9975

3. Persistence image resolution (dim=3, tau=31 fixed)
  n_pixels=10  →  F1=0.9925
  n_pixels=20  →  F1=0.9925
  n_pixels=30  →  F1=0.9926

Summary — locked values marked with ★
  dim    :  dim=2:0.9531  ★dim=3:0.9925   dim=4:0.9951  
  tau    :  tau=15:0.9654  ★tau=31:0.9925   tau=50:0.9975  
  pixels :  n_pix=10:0.9925  ★n_pix=20:0.9925   n_pix=30:0.9926  


In [7]:
# Cell 5 — Final baseline comparison table
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score
from tda_detect.features import TDAFeatureExtractor

# ── Local signal generators (no global scope dependency) ─────────────────────
t_local = np.linspace(0, 4*np.pi, 500)

def make_normal_local(n, seed=123):
    rng = np.random.default_rng(seed)
    return [np.sin(t_local) + rng.normal(0, 0.05, 500) for _ in range(n)]

def make_anomaly_local(n, seed=123):
    rng  = np.random.default_rng(seed)
    sigs = []
    for _ in range(n):
        anom_type = rng.choice(["phase_break", "freq_mod", "dual_freq"])
        noise = rng.normal(0, 0.05, 500)
        if anom_type == "phase_break":
            sig = np.sin(t_local) + noise
            bp  = int(rng.integers(150, 350))
            sig[bp:] = np.sin(t_local[bp:] + np.pi*0.7) + noise[bp:]
        elif anom_type == "freq_mod":
            mod = 1.0 + 0.3*np.sin(t_local*0.4)
            sig = np.sin(t_local)*mod + noise
            sig = sig / sig.std() * (np.sin(t_local)+noise).std()
        else:
            sig = 0.7*np.sin(t_local) + 0.7*np.sin(1.03*t_local) + noise
            sig = sig / sig.std() * (np.sin(t_local)+noise).std()
        sigs.append(sig)
    return sigs

# ── Build test set ────────────────────────────────────────────────────────────
print("Building test set...")
test_normal  = make_normal_local(100, seed=123)
test_anomaly = make_anomaly_local(100, seed=123)
test_signals = test_normal + test_anomaly
y_test       = np.array([0]*100 + [1]*100)
print(f"  Test set: {len(test_signals)} signals (100 normal + 100 anomaly)")

# ── Z-score baseline ──────────────────────────────────────────────────────────
def zscore_predict(signals, threshold=2.5):
    preds = []
    for sig in signals:
        z = np.abs((sig - sig.mean()) / sig.std())
        preds.append(1 if z.max() > threshold else 0)
    return np.array(preds)

best_f1_z, best_thr_z = -1, 0
for thr in np.linspace(1.0, 5.0, 100):
    preds = zscore_predict(test_signals, threshold=thr)
    f = f1_score(y_test, preds, zero_division=0)
    if f > best_f1_z:
        best_f1_z, best_thr_z = f, thr
z_preds = zscore_predict(test_signals, threshold=best_thr_z)
z_f1    = f1_score(y_test, z_preds, zero_division=0)
z_prec  = precision_score(y_test, z_preds, zero_division=0)
z_rec   = recall_score(y_test, z_preds, zero_division=0)
print(f"  ✓ Z-score baseline done        F1={z_f1:.4f}")

# ── Rolling std baseline ──────────────────────────────────────────────────────
def rolling_std_predict(signals, window=50, threshold=0.15):
    preds = []
    for sig in signals:
        stds = [sig[i:i+window].std() for i in range(0, len(sig)-window, window)]
        preds.append(1 if max(stds) > threshold else 0)
    return np.array(preds)

best_f1_r, best_thr_r = -1, 0
for thr in np.linspace(0.05, 0.5, 100):
    preds = rolling_std_predict(test_signals, threshold=thr)
    f = f1_score(y_test, preds, zero_division=0)
    if f > best_f1_r:
        best_f1_r, best_thr_r = f, thr
r_preds = rolling_std_predict(test_signals, threshold=best_thr_r)
r_f1    = f1_score(y_test, r_preds, zero_division=0)
r_prec  = precision_score(y_test, r_preds, zero_division=0)
r_rec   = recall_score(y_test, r_preds, zero_division=0)
print(f"  ✓ Rolling std baseline done    F1={r_f1:.4f}")

# ── Raw IsolationForest (no TDA) ──────────────────────────────────────────────
X_test_raw  = np.array(test_signals)
X_train_raw = np.array(make_normal_local(160, seed=0))
clf_raw     = IsolationForest(n_estimators=200, contamination="auto",
                               random_state=42, n_jobs=-1)
clf_raw.fit(X_train_raw)
scores_raw  = clf_raw.score_samples(X_test_raw)
candidates  = np.linspace(scores_raw.min(), scores_raw.max(), 300)
best_f1_raw, best_thr_raw = -1, candidates[0]
for thr in candidates:
    preds = (scores_raw < thr).astype(int)
    f = f1_score(y_test, preds, zero_division=0)
    if f > best_f1_raw:
        best_f1_raw, best_thr_raw = f, thr
raw_preds = (scores_raw < best_thr_raw).astype(int)
raw_f1    = f1_score(y_test, raw_preds, zero_division=0)
raw_prec  = precision_score(y_test, raw_preds, zero_division=0)
raw_rec   = recall_score(y_test, raw_preds, zero_division=0)
print(f"  ✓ Raw IsolationForest done     F1={raw_f1:.4f}")

# ── TDA-IsolationForest ───────────────────────────────────────────────────────
print("  Extracting TDA features for test set (takes a few minutes)...")
ext         = TDAFeatureExtractor()
X_test_tda  = np.array([ext.transform(s) for s in test_signals])
X_train_tda = np.array([ext.transform(s) for s in make_normal_local(160, seed=0)])
clf_tda     = IsolationForest(n_estimators=200, contamination="auto",
                               random_state=42, n_jobs=-1)
clf_tda.fit(X_train_tda)
scores_tda  = clf_tda.score_samples(X_test_tda)
candidates  = np.linspace(scores_tda.min(), scores_tda.max(), 300)
best_f1_tda, best_thr_tda = -1, candidates[0]
for thr in candidates:
    preds = (scores_tda < thr).astype(int)
    f = f1_score(y_test, preds, zero_division=0)
    if f > best_f1_tda:
        best_f1_tda, best_thr_tda = f, thr
tda_preds = (scores_tda < best_thr_tda).astype(int)
tda_f1    = f1_score(y_test, tda_preds, zero_division=0)
tda_prec  = precision_score(y_test, tda_preds, zero_division=0)
tda_rec   = recall_score(y_test, tda_preds, zero_division=0)
print(f"  ✓ TDA-IsolationForest done     F1={tda_f1:.4f}")

# ── Print table ───────────────────────────────────────────────────────────────
print(f"\n{'=' * 65}")
print(f"  Final Baseline Comparison (n=200: 100 normal + 100 anomaly)")
print(f"{'=' * 65}")
print(f"  {'Model':<30} {'F1':>8} {'Precision':>10} {'Recall':>8} {'vs TDA':>8}")
print(f"  {'-'*62}")

models = [
    ("Z-score",                  z_f1,   z_prec,   z_rec),
    ("Rolling Std",              r_f1,   r_prec,   r_rec),
    ("IsolationForest (raw)",    raw_f1, raw_prec, raw_rec),
    ("IsolationForest (TDA) ★", tda_f1, tda_prec, tda_rec),
]
for name, f1, prec, rec in models:
    if name.endswith("★"):
        diff = "+0.0%"
    else:
        diff = f"{((f1 - tda_f1) / max(tda_f1, 1e-9)) * 100:+.1f}%"
    print(f"  {name:<30} {f1:>8.4f} {prec:>10.4f} {rec:>8.4f} {diff:>8}")

print(f"{'=' * 65}")
print(f"  ★ = proposed method")

Building test set...
  Test set: 200 signals (100 normal + 100 anomaly)
  ✓ Z-score baseline done        F1=0.6667
  ✓ Rolling std baseline done    F1=0.8066
  ✓ Raw IsolationForest done     F1=1.0000
  Extracting TDA features for test set (takes a few minutes)...
  ✓ TDA-IsolationForest done     F1=0.9950

  Final Baseline Comparison (n=200: 100 normal + 100 anomaly)
  Model                                F1  Precision   Recall   vs TDA
  --------------------------------------------------------------
  Z-score                          0.6667     0.5000   1.0000   -33.0%
  Rolling Std                      0.8066     0.9012   0.7300   -18.9%
  IsolationForest (raw)            1.0000     1.0000   1.0000    +0.5%
  IsolationForest (TDA) ★          0.9950     0.9901   1.0000    +0.0%
  ★ = proposed method
